## Step 2 of the seminar

starting from the meta-algorithm, we start going in deep performing a Q-learning. The program still uses 1 agent and one principal

The MDP remains the same (principla_agent_mdp.py), but the optimizations use new implementations

In [1]:
# meta algorithm with Q_leraning
import numpy as np
from principal_agent_mdp import PrincipalAgentMDP
from agent_qlearn import Agent
from principal_isa import Principal

In [2]:


GAMMA      = 1.0
N_META     = 3
N_EPISODES = 5000
ALPHA      = 0.1
EPSILON    = 0.1

mdp   = PrincipalAgentMDP(gamma=GAMMA)
agent = Agent(mdp, alpha=ALPHA, epsilon=EPSILON)

r_p = np.zeros((mdp.n_states, mdp.n_outcomes))
for s in range(mdp.n_states):
    r_p[s, 0] = 14/9
    r_p[s, 1] = 0.0

principal = Principal(mdp, r_p, alpha=ALPHA, epsilon=EPSILON)

# rho: s -> tuple (b(L), b(R)) — matches Principal output format
rho = {s: (0.0, 0.0) for s in range(mdp.n_states)}

for meta in range(N_META):
    print(f"\n── Meta iteration {meta+1} ──")


    # ── Inner: agent ───────────────────────────────────────────
    agent.reset()
    agent_returns = []

    for episode in range(N_EPISODES):
        s = mdp.s0
        agent_return = 0.0
        t = 0

        while not mdp.is_terminal(s):
            b = rho[s]
            a = agent.act(s, b)
            o = mdp.sample_outcome(s, a)
            s_next = mdp.T(s, o)
            b_next = rho[s_next]

            r_a = mdp.R_agent(s, a, b, o)
            agent_return += (GAMMA ** t) * r_a
            t += 1

            agent.update(s, a, o, s_next, b_next)
            s = s_next

        agent_returns.append(agent_return)

    print(f"Q_bar:\n{agent.Q_bar}")

    # ── Outer: principal ────────────────────────────────────────
    principal.reset()
    principal_returns = []
    Q_bar = agent.get_Q_bar()
    contract_table = {}
    for s in range(mdp.n_states):
        for a_p in range(mdp.n_actions):
            contract_table[(s, a_p)] = principal.find_best_contract(s, a_p, Q_bar)

    for episode in range(N_EPISODES):
        s = mdp.s0
        principal_return = 0.0
        t = 0

        while not mdp.is_terminal(s):

            if np.random.rand() < EPSILON:
                a_p = np.random.randint(mdp.n_actions)
            else:
                a_p = int(np.argmax(principal.q[s]))

            b = contract_table[(s, a_p)]
            o = mdp.sample_outcome(s, a_p)
            s_next = mdp.T(s, o)

            r_p = mdp.R_principal(s, b, o)
            principal_return += (GAMMA ** t) * r_p
            t += 1

            principal.update(s, a_p, b, o, s_next)
            s = s_next

        principal_returns.append(principal_return)
    
    

    # extract rho
    for s in range(mdp.n_states):
        best_a_p = int(np.argmax(principal.q[s]))
        rho[s]   = contract_table[(s, best_a_p)]

    print(f"Principal q:\n{principal.q}")
    for s, name in enumerate(['s0', 'sL', 'sR']):
        print(f"rho({name}) = {rho[s]}")
    print("Agent avg return:", np.mean(agent_returns))
    print("Principal avg return:", np.mean(principal_returns))


── Meta iteration 1 ──
Q_bar:
[[-0.8  0. ]
 [ 0.   0. ]
 [ 0.   0. ]]
Principal q:
[[0.4987424  0.00693458]
 [0.         0.        ]
 [0.         0.        ]]
rho(s0) = (np.float64(1.0), np.float64(0.0))
rho(sL) = (np.float64(0.0), np.float64(0.0))
rho(sR) = (np.float64(0.0), np.float64(0.0))
Agent avg return: -0.043039999999999995
Principal avg return: 0.4811999999999999

── Meta iteration 2 ──
Q_bar:
[[-0.8  0. ]
 [ 0.   0. ]
 [ 0.   0. ]]
Principal q:
[[0.54025709 0.24115596]
 [0.         0.        ]
 [0.         0.        ]]
rho(s0) = (np.float64(1.0), np.float64(0.0))
rho(sL) = (np.float64(0.0), np.float64(0.0))
rho(sR) = (np.float64(0.0), np.float64(0.0))
Agent avg return: 0.09903999999999995
Principal avg return: 0.4855333333333333

── Meta iteration 3 ──
Q_bar:
[[-0.8  0. ]
 [ 0.   0. ]
 [ 0.   0. ]]
Principal q:
[[0.47828241 0.17603016]
 [0.         0.        ]
 [0.         0.        ]]
rho(s0) = (np.float64(1.0), np.float64(0.0))
rho(sL) = (np.float64(0.0), np.float64(0.0))
